In [4]:
import boto3

bucket = 'rsugumar-insurance-prediction-ml-project-2026-01'
prefix = 'insurance-data'
local_file = 'insurance_pre.csv'

s3 = boto3.client('s3')

s3_key = f'{prefix}/{local_file}'

s3.upload_file(local_file, bucket, s3_key)

s3_path = f's3://{bucket}/{s3_key}'

from utils import append_to_file
append_to_file("output.txt", "s3-path:", s3_path)

print("✅ File uploaded successfully!")
print(f"S3 Path: {s3_path}")

✅ File uploaded successfully!
S3 Path: s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/insurance_pre.csv


In [1]:
import sys

print(sys.executable)

/opt/conda/bin/python


In [2]:
import sys

!{sys.executable} -m pip install --no-cache-dir "sagemaker==2.257.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 229.0 MB/s  0:00:00
  Attempting uninstall: sagemaker
    Found existing installation: sagemaker 2.256.0
    Uninstalling sagemaker-2.256.0:
      Successfully uninstalled sagemaker-2.256.0


In [3]:
import sys

!{sys.executable} -m pip show sagemaker

Name: sagemaker
Version: 2.257.6
Summary: Open source library for training and deploying models on Amazon SageMaker.
Home-page: https://github.com/aws/sagemaker-python-sdk
Author: Amazon Web Services
Author-email: 
License: 
Location: /opt/conda/lib/python3.12/site-packages
Requires: attrs, boto3, cloudpickle, docker, fastapi, google-pasta, graphene, importlib-metadata, jsonschema, numpy, omegaconf, packaging, pandas, pathos, platformdirs, protobuf, psutil, pytz, pyyaml, requests, sagemaker-core, schema, smdebug-rulesconfig, tblib, tqdm, urllib3, uvicorn
Required-by: 


In [37]:
import sys

!{sys.executable} -m pip install --force-reinstall --no-cache-dir "sagemaker==2.256.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 172.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 234.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 152.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 3.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 187.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 553.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 238.2 MB/s  0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144590 sha256=3d9704cd00a82a77d1ddcda1e5a1ed1e7f2f985bea508730fc8efdd86b036ce2
  Stored in directory: /tmp/pip-ephem-wheel-cache-d3f0btcb/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfull

In [4]:
from sagemaker.sklearn.estimator import SKLearn

print("✅ SKLearn import successful")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


✅ SKLearn import successful


In [7]:
print(role)

arn:aws:iam::979604620994:role/service-role/AmazonSageMaker-ExecutionRole-20260823T122235


Yes — now we know the exact SageMaker execution role being used:

AmazonSageMaker-ExecutionRole-20260823T122235

Your previous error means this role is being denied when SageMaker tries to upload source.tar.gz into your S3 bucket.

What we need to do

We need to give this role permission to upload to:

s3://insurance-prediction-bucket/

At minimum, it needs s3:PutObject permission for the bucket's objects.

In AWS Console
Open IAM
Click Roles
Search for:
AmazonSageMaker-ExecutionRole-20260823T122235
Open that role.
Go to Permissions.
Look for an existing S3 policy.

If there is already an S3 policy attached, don't create another one yet. Show me the permissions page.

If there isn't an appropriate S3 policy, we can add one specifically for your:

insurance-prediction-bucket
Important

Your role itself is valid:

arn:aws:iam::979604620994:role/service-role/AmazonSageMaker-ExecutionRole-20260823T122235

And your error is specifically S3 PutObject AccessDenied, not a SageMaker quota problem.

=====

The policy should be:

{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "s3:ListBucket"
            ],
            "Effect": "Allow",
            "Resource": [
                "arn:aws:s3:::SageMaker",
                "arn:aws:s3:::insurance-prediction-bucket"
            ]
        },
        {
            "Action": [
                "s3:GetObject",
                "s3:PutObject",
                "s3:DeleteObject"
            ],
            "Effect": "Allow",
            "Resource": [
                "arn:aws:s3:::SageMaker/*",
                "arn:aws:s3:::insurance-prediction-bucket/*"
            ]
        }
    ]
}

In [11]:
import boto3

s3 = boto3.client("s3")

s3.put_object(
    Bucket="rsugumar-insurance-prediction-ml-project-2026-01",
    Key="test-sagemaker-permission.txt",
    Body=b"test"
)

print("✅ S3 PutObject works")

✅ S3 PutObject works


In [13]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role
import sagemaker
from utils import append_to_file

session = sagemaker.Session()

role = get_execution_role()

bucket = "rsugumar-insurance-prediction-ml-project-2026-01"
prefix = "insurance-data"

sklearn_estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.xlarge",
    framework_version="1.2-1",
    py_version="py3",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session
)

append_to_file("output.txt", "framework_version 1.2-1")

# Start training
sklearn_estimator.fit({
    "train": f"s3://{bucket}/{prefix}/"
})

image_uri = sklearn_estimator.training_image_uri()

append_to_file("output.txt", "Image URI", image_uri)

model_s3_uri = sklearn_estimator.model_data

append_to_file("output.txt", "model_s3_uri", model_s3_uri)

print("Model S3 URI:", model_s3_uri)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-scikit-learn-2026-08-24-05-42-22-834


2026-08-24 05:42:24 Starting - Starting the training job...
2026-08-24 05:42:40 Starting - Preparing the instances for training...
2026-08-24 05:43:21 Downloading - Downloading the training image......
2026-08-24 05:44:23 Training - Training image download completed. Training in progress.
2026-08-24 05:44:23 Uploading - Uploading generated training model/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-08-24 05:44:15,555 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2026-08-24 05:44:15,560 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-08-24 05:44:15,563 sagemaker-training-toolkit INFO     No Neurons detected (norma

Yes! 🎉 Your SageMaker training job has started successfully.

Your screenshot shows:

INFO:sagemaker:Creating training-job with name:
sagemaker-scikit-learn-2026-08-24-05-42-22-834

and:

2026-08-24 05:42:24 Starting - Starting the training job...
Current status
Step	Status
SageMaker SDK / SKLearn	✅
S3 permissions	✅
S3 source upload	✅
Training job creation	✅
ml.m5.xlarge training	🟢 Starting
train.py execution	⏳ Next

The pink/red-looking messages at the top are informational SageMaker telemetry messages, not an error.

What you should do now

Don't stop the cell. Don't restart the kernel.

Just wait for the training job to finish.

For a small insurance dataset, it may take several minutes. When it finishes, you should eventually see something like:

Completed - Training job completed

and then:

Model S3 URI: s3://...

Once you see the final output, send me the screenshot. We'll then check the model S3 URI and proceed to the next step.

🎉 Excellent — your SageMaker training completed successfully!

The most important line is:

2026-08-24 05:44:17,254 sagemaker-containers INFO     Reporting training SUCCESS

Your train.py also successfully:

Loaded the dataset: 1,338 rows × 6 columns
Created dummy variables:
sex_male, smoker_yes
Achieved R² = 0.85784268869665
Saved the trained model as:
/opt/ml/model/model.joblib

The log confirms the training job used ml.m5.xlarge and successfully invoked train.py.

| Item           | Result                |
| -------------- | --------------------- |
| Training job   | ✅ Successful          |
| Instance       | `ml.m5.xlarge`        |
| Dataset        | 1,338 × 6             |
| R² score       | **0.85784 (~85.78%)** |
| Model file     | `model.joblib`        |
| Model saved    | ✅                     |
| Training error | ❌ None                |


🎉 Excellent! Your SageMaker training job is fully completed.

Your final result is:

Training job completed
Training seconds: 94
Billable seconds: 94

And your trained model is now stored in S3:

s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz

======

project status

| Step                    | Status               |
| ----------------------- | -------------------- |
| JupyterLab              | ✅                    |
| SageMaker SDK / SKLearn | ✅                    |
| S3 permissions          | ✅                    |
| Training job            | ✅                    |
| Instance                | `ml.m5.xlarge`       |
| Training time           | **94 seconds**       |
| Model accuracy (R²)     | **0.85784 / 85.78%** |
| Model artifact          | ✅ Saved to S3        |
| `model.tar.gz`          | ✅ Ready              |

====

Most important: training cost

Your job reports:

94 billable seconds ≈ 1 minute 34 seconds.

So you only consumed about:

0.0261 training hours.

If your account is eligible for the SageMaker training Free Tier and has remaining hours, this usage can be covered by that allocation.

Your model location

Keep this value — we'll need it for deployment:

model_s3_uri = "s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz"

===

Do not run the training again. The model is successfully trained and saved.

The next stage is deploying this model.tar.gz as a SageMaker endpoint so you can send values such as age, BMI, children, sex, and smoker status and receive the predicted insurance charge.

Input data:

s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/

Output model:

s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz

IAM role:

AmazonSageMaker-ExecutionRole-20260823T122235

Encryption: Default encryption

So the training stage is finished. 🎯

====

project pipeline so far

insurance_pre.csv
       ↓
      S3
       ↓
SageMaker Training
       ↓
ml.m5.xlarge
       ↓
train.py
       ↓
R² = 0.85784
       ↓
model.joblib
       ↓
model.tar.gz
       ↓
S3 ✅

Next step: create the SageMaker model

Now we can use this model artifact to create a SageMaker Model and then deploy it for prediction.

Your model URI is:

model_s3_uri = "s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz

arn:aws:sagemaker:us-east-1:979604620994:training-job/sagemaker-scikit-learn-2026-08-24-05-42-22-834

Don't retrain the model. You are ready for the deployment/model creation stage.

In [14]:
image_uri = sklearn_estimator.training_image_uri()

append_to_file("output.txt", "Image URI", image_uri)

model_s3_uri = sklearn_estimator.model_data

append_to_file("output.txt", "model_s3_uri", model_s3_uri)

print("Model S3 URI:", model_s3_uri)

Model S3 URI: s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz


In [15]:
from sagemaker.sklearn.model import SKLearnModel
model = SKLearnModel(
    model_data="s3://rsugumar-insurance-prediction-ml-project-2026-01/insurance-data/output/sagemaker-scikit-learn-2026-08-24-05-42-22-834/output/model.tar.gz",  # <- from training
    role=role,
    entry_point="inference.py",                         # ✅ NEW inference script
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=session
)

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: SKLearnModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


package/use this inference.py with the trained model and create the SageMaker inference endpoint.


===

insurance_pre.csv
       ↓
Training
       ↓
model.joblib
       ↓
model.tar.gz
       ↓
S3
       ↓
       ⭐ INFERENCE
       ↓
New customer data
       ↓
Predicted insurance charge

====

Inference has 3 steps

| Step               | What happens                                      |
| ------------------ | ------------------------------------------------- |
| 1. Create Model    | Load `model.tar.gz`                               |
| 2. Deploy Endpoint | Put the model behind an endpoint                  |
| 3. Predict         | Send new customer data → receive insurance charge |


In [16]:
predictor = model.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1,
    endpoint_name="insurance-charge-sugu-endpoint"
)

INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2026-08-24-07-05-02-957
INFO:sagemaker:Creating endpoint-config with name insurance-charge-sugu-endpoint
INFO:sagemaker:Creating endpoint with name insurance-charge-sugu-endpoint


------!

/opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: SKLearnPredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


| Code                                         | Meaning                                    |
| -------------------------------------------- | ------------------------------------------ |
| `model.deploy()`                             | Deploys your trained model                 |
| `instance_type="ml.m5.large"`                | Uses an `ml.m5.large` inference instance   |
| `initial_instance_count=1`                   | Starts 1 inference instance                |
| `endpoint_name="insurance-charge-endpoint4"` | Name of your prediction endpoint           |
| `predictor`                                  | Object you use to send prediction requests |


complete inference flow
---------------------

model.tar.gz
     ↓
SageMaker Model
     ↓
model.deploy()
     ↓
ml.m5.large
     ↓
insurance-charge-endpoint4
     ↓
predictor
     ↓
New customer data
     ↓
Insurance charge prediction